# 03. 根拠の抽出と検証

本アプリの中核。**Ollama が無くても最後まで動く**（フィクスチャを使う）。

流れ:

```
トレース ──> build_candidates()  根拠候補（eid 付き）
             ↓
         Extractor  「候補の eid からしか選べない」制約付きで仮説 JSON を生成
             ↓
         Verifier   実在検証・包含チェック（C ⊆ B）
             ↓
         レポート
```

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from biomni_hypo.config import Settings
from biomni_hypo.fixtures import SAMPLE_QUESTION, SAMPLE_SOLUTION, sample_steps

settings = Settings(offline_mode=True)
steps = sample_steps()

for s in steps:
    print(f"[{s.idx}] {s.kind.value:12s} {(s.code or s.text)[:70].strip().splitlines()[0] if (s.code or s.text) else ''}")

## 1. 根拠候補を作る

トレースから機械的に抽出する。**ここに無いものは仮説の根拠になれない。**
抜粋（excerpt）は必ず実テキストから切り出す。LLM には書かせない。

In [ ]:
from biomni_hypo.extractor import build_candidates

candidates = build_candidates(steps)
for c in candidates:
    print(f"{c.eid:4s} {c.kind.value:12s} {c.identifier:32s} step={c.step_idx}  {c.excerpt[:60]}")

## 2. Extractor に渡すプロンプト

候補は ID 付きの表として渡す。「リストに無い ID を書くな」を明示する。

In [ ]:
from biomni_hypo.extractor import HypothesisExtractor
from biomni_hypo.fixtures import FakeLLM, fake_extraction_response

extractor = HypothesisExtractor(settings, llm=FakeLLM(fake_extraction_response()))
prompt = extractor.build_prompt(SAMPLE_QUESTION, SAMPLE_SOLUTION, steps, candidates)
print(prompt[:2500])

## 3. 抽出（まずフィクスチャで）

In [ ]:
result = extractor.extract(SAMPLE_QUESTION, steps, SAMPLE_SOLUTION)
print("パース成功:", result.ok, result.parse_error)
for h in result.hypotheses:
    print(f"\n■ {h.statement}")
    print(f"  確度 {h.confidence} / 新規性 {h.novelty} / 根拠 {len(h.evidence)} 件")
    for ev in h.evidence:
        print(f"    - [{ev.eid}] {ev.identifier} ({ev.stance.value}) {ev.why[:50]}")

## 4. 幻覚した根拠 ID は落ちる

LLM が存在しない `E999` を書いてきた場合。仮説は残すが、その根拠だけを捨てる。

In [ ]:
from biomni_hypo.extractor import parse_response

bad = parse_response(fake_extraction_response(include_unknown_eid=True), candidates)
print("破棄した未知 eid:", bad.unknown_eids)
print("仮説に残った eid:", [ev.eid for h in bad.hypotheses for ev in h.evidence])
assert "E999" not in [ev.eid for h in bad.hypotheses for ev in h.evidence]
print("✅ 存在しない根拠は仮説に入らない")

## 5. 検証（Verifier）

LLM を一切使わない。トレースとの突き合わせと外部 API 照会だけ。

最重要は **包含チェック（C ⊆ B）**: 「主張を支えた」根拠が「実行結果に現れた」ものに
含まれていなければ、トレースに存在しない出所なので無条件に落とす。

In [ ]:
from biomni_hypo.verifier import EvidenceVerifier

verifier = EvidenceVerifier(offline=True)   # オフラインでは PMID 照会をスキップ
supported, unsupported, report = verifier.verify_run(result.hypotheses, steps)

print("裏付けあり:", len(supported), "/ 未裏付け:", len(unsupported))
print("検証結果  :", report.summary.model_dump())
for h in supported:
    for ev in h.evidence:
        print(f"  {ev.verification_status.value:16s} {ev.identifier:28s} {ev.verification_note}")

In [ ]:
# 捏造された識別子を混ぜて、確実に落ちることを確認する
from biomni_hypo.schemas import Evidence, Hypothesis, ResourceKind

fake_h = Hypothesis(
    statement="捏造された根拠だけの仮説",
    evidence=[Evidence(eid="X1", kind=ResourceKind.DB_RECORD, identifier="rs00000000", step_idx=2)],
)
sup, unsup, rep = EvidenceVerifier(offline=True).verify_run([fake_h], steps)
print("裏付けあり:", len(sup), "/ 未裏付け:", len(unsup))
print("失敗した引用:", [(f.identifier, f.reason) for f in rep.failed])

## 6. PMID の実在検証（オンライン）

ネットワークがあれば実際に NCBI へ問い合わせる。
存在しない PMID を混ぜて、落ちることを確認する。

In [ ]:
from biomni_hypo.schemas import Evidence, ResourceKind
from biomni_hypo.verifier import EvidenceVerifier, TraceIndex

online = EvidenceVerifier(offline=False)
index = TraceIndex.from_steps(steps)

for pmid in ["PMID:17529967", "PMID:99999999"]:
    ev = Evidence(eid="t", kind=ResourceKind.LITERATURE, identifier=pmid, step_idx=2)
    status, note = online.verify_evidence(ev, index)
    print(f"{pmid}: {status.value:16s} {note[:90]}")

`PMID:99999999` は「トレースに出現しない識別子」としてまず落ちる（外部照会に行く前）。
包含チェックが先に効いているのが正しい挙動。

## 7. レポート

使用データとライセンスのセクションが自動生成される（CC BY 系の帰属義務を機械的に満たす）。

In [ ]:
from biomni_hypo.pipeline import collect_resources
from biomni_hypo.policy import ResourcePolicy
from biomni_hypo.report import to_markdown
from biomni_hypo.schemas import RunResult

policy = ResourcePolicy.load(settings.policy_path)
run = RunResult(
    id="nb03", question=SAMPLE_QUESTION, status="succeeded",
    config=settings.to_run_config(policy_version=policy.version),
    steps=steps, solution_text=SAMPLE_SOLUTION,
    hypotheses=supported, unsupported_ideas=unsupported,
    failed_citations=report.failed, verification=report.summary,
    resources_used=collect_resources(steps, policy),
)
md_text = to_markdown(run, include_trace=False)
print(md_text)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(md_text))

## 8. 実際の Ollama で抽出させる（任意）

フィクスチャの代わりに本物のモデルを使う。スキーマ準拠が崩れる場合は
`settings.max_hypotheses` を下げるか、より大きいモデルを試す。

In [ ]:
from biomni_hypo.llm import ollama_status

if ollama_status(settings.ollama_base_url).reachable:
    real = HypothesisExtractor(settings)      # llm を渡さなければ Ollama を使う
    real_result = real.extract(SAMPLE_QUESTION, steps, SAMPLE_SOLUTION)
    print("パース成功:", real_result.ok, real_result.parse_error)
    print("未知 eid  :", sorted(set(real_result.unknown_eids)))
    for h in real_result.hypotheses:
        print(f"\n■ {h.statement}")
        for ev in h.evidence:
            print(f"    - [{ev.eid}] {ev.identifier} ({ev.stance.value})")
else:
    print("Ollama 未起動のためスキップ")